# 1. 모델 학습 · 예측 저장

`data/` 를 읽어 **walk-forward OOS**(과거→미래, test=미래 구간)로 학습하고,
**모델별 예측**을 `result/predictions/<model>.csv` 에 저장합니다. **타깃은 MC 이론가**(공정가치 아님).
정통 DeepONet 재설계: **branch = 시장상태(바스켓 vol·corr + 수익률곡선), trunk = 계약(전체 STRK 스케줄 + 배리어 + 쿠폰 + 만기)**.

- **벤치마크(직접·단일 단계)**: Ridge/GBM/LightGBM/XGBoost/CatBoost — `ml.csv` 전 특성으로 MC 이론가를 바로 예측
- **DeepONet-Curve 하이브리드(2단계)**: 1D-CNN 곡선 branch — stage-1 MC 앵커(MC_hat) + stage-2 잔차(MC−MC_hat). 앵커 loss MSE/L1/MAPE + stage-2 DeepONet 변형(`s2don`)
- **XGBoost 하이브리드(2단계)**: 앵커·잔차 모두 XGB (DeepONet 앵커와 동일 입력 = 순수 아키텍처 비교)

2단계 하이브리드 예측 csv에는 stage별 R² 계산용 컬럼(`mc_pred`, `resid_pred` 등)이 함께 저장됩니다.

In [1]:
from util import utils, file_manager as fm
from module import data
from model import registry

cfg = utils.load_config()
utils.set_seed(cfg["seed"])
D = data.load(cfg)
print(f"loaded | rows {D.n} | folds {len(D.WF)} | device {D.DEV}")
print("discovered models:", [n for n, _ in registry.discover()])

loaded | rows 23151 | folds 4 | device cuda


discovered models: ['benchmark', 'deeponet']


In [2]:
# model/*.py 자동발견·실행 (새 .py 를 model/ 에 넣으면 자동 편입)
preds = registry.run_all(D, cfg)

for name, df in preds.items():
    path = fm.prediction(name)
    df.to_csv(path, index=False)
    print(f"saved {name:24s} rows {len(df):5d} -> {path.relative_to(fm.ROOT)}")
print("\nDONE")

[model.benchmark] -> ['bench_ridge', 'bench_gbm', 'bench_lgbm', 'bench_xgboost', 'bench_catboost', 'xgb_hybrid']


[model.deeponet] -> ['deeponet_hybrid', 'deeponet_hybrid_s2don', 'deeponet_hybrid_l1', 'deeponet_hybrid_mape']
saved bench_ridge              rows  9261 -> result\predictions\bench_ridge.csv
saved bench_gbm                rows  9261 -> result\predictions\bench_gbm.csv
saved bench_lgbm               rows  9261 -> result\predictions\bench_lgbm.csv
saved bench_xgboost            rows  9261 -> result\predictions\bench_xgboost.csv
saved bench_catboost           rows  9261 -> result\predictions\bench_catboost.csv
saved xgb_hybrid               rows  9261 -> result\predictions\xgb_hybrid.csv
saved deeponet_hybrid          rows  9261 -> result\predictions\deeponet_hybrid.csv
saved deeponet_hybrid_s2don    rows  9261 -> result\predictions\deeponet_hybrid_s2don.csv


saved deeponet_hybrid_l1       rows  9261 -> result\predictions\deeponet_hybrid_l1.csv
saved deeponet_hybrid_mape     rows  9261 -> result\predictions\deeponet_hybrid_mape.csv

DONE
